# Linear Regression Du Bao Luong Va Nhu Cau Tuyen Dung (opencode)

Notebook nay chi dung **Linear Regression thuan** de hoc va hieu cach mo hinh tuyen tinh hoat dong.

Muc tieu:

- Bai toan 1: du bao muc luong CNTT tu cac feature nhu source, location, seniority, experience, skills.
- Bai toan 2: du bao nhu cau tuyen dung bang `job_count` theo tuan, source, skill.
- Tap trung vao cong thuc `y = b0 + b1*x1 + b2*x2 + ...`, coefficient, error, va han che cua Linear Regression.

Luu y: cac buoc nhu `OneHotEncoder`, `SimpleImputer`, `Pipeline` chi la tien xu ly data. Model hoc van la `sklearn.linear_model.LinearRegression`.

## 0. Setup

Import thu vien can thiet. Notebook nay khong dung Ridge, Lasso, RandomForest, XGBoost, ARIMA, Prophet hay Poisson.

In [ ]:
from pathlib import Path
from collections import Counter
import ast
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "analysis" / "salary_analysis_clean.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing data/analysis/salary_analysis_clean.csv")


REPO_ROOT = find_repo_root()
SALARY_PATH = REPO_ROOT / "data" / "analysis" / "salary_analysis_clean.csv"
REPORTS_DIR = REPO_ROOT / "data" / "reports"
DEMAND_PATH = REPORTS_DIR / "demand_by_week_skill.csv"

print(f"Repo root: {REPO_ROOT}")
print(f"Salary file: {SALARY_PATH}")
print(f"Demand report: {DEMAND_PATH}")

## 1. Helper Functions

Cac function nay chi de lam sach data: convert luong ve VND/thang, chuan hoa location, chuan hoa skill, va tao skill flags.

Quan trong: khong dua `salary_raw`, `salary_min`, `salary_max`, `salary_midpoint`, `salary_currency` vao feature vi do la leakage.

In [ ]:
USD_TO_VND = 26_000


def strip_accents(value):
    if pd.isna(value):
        return ""
    value = str(value).replace("Đ", "D").replace("đ", "d")
    return "".join(character for character in unicodedata.normalize("NFD", value) if unicodedata.category(character) != "Mn")


def fold_text(value):
    return strip_accents(value).casefold().strip()


def clean_text_series(series):
    cleaned = series.astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    return cleaned.mask(cleaned.eq(""), pd.NA)


def normalize_location(value):
    folded = fold_text(value)
    if any(token in folded for token in ["hcm", "ho chi minh", "thanh pho ho chi minh", "sai gon"]):
        return "Ho Chi Minh"
    if "ha noi" in folded or "hanoi" in folded:
        return "Ha Noi"
    if "da nang" in folded or "danang" in folded:
        return "Da Nang"
    if not folded or folded == "nan":
        return "Unknown"
    return str(value).strip()


def to_listish(value):
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed if str(item).strip()]
        except (ValueError, SyntaxError):
            pass
    return [part.strip() for part in text.replace(";", ",").split(",") if part.strip()]


skill_alias = {
    "js": "javascript",
    "reactjs": "react",
    "react.js": "react",
    "vuejs": "vue",
    "vue.js": "vue",
    "node": "node.js",
    "nodejs": "node.js",
    "golang": "go",
    "postgres": "postgresql",
    "k8s": "kubernetes",
    "ts": "typescript",
    "csharp": "c#",
    "dotnet": ".net",
    "html5": "html",
    "css3": "css",
}


def normalize_skill(value):
    folded = " ".join(fold_text(value).replace("&", " and " ).split())
    return skill_alias.get(folded, folded)


def skill_column_name(skill):
    safe = "".join(character if character.isalnum() else "_" for character in skill)
    safe = "_".join(part for part in safe.split("_") if part)
    return f"skill_{safe}"


def has_annual_salary_signal(text):
    words = set(str(text).replace("/", " " ).replace("-", " " ).split())
    return bool(words & {"year", "annual", "annually", "nam"})


def prepare_salary_data(salary):
    frame = salary.copy()

    for column in ["source", "title", "company", "location", "salary_raw", "salary_currency", "salary_period", "seniority", "work_mode", "skills"]:
        if column in frame.columns:
            frame[column] = clean_text_series(frame[column])

    for column in ["salary_min", "salary_max", "salary_midpoint", "experience_min", "experience_max"]:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

    if "salary_midpoint" not in frame.columns or frame["salary_midpoint"].isna().all():
        frame["salary_midpoint"] = frame[["salary_min", "salary_max"]].mean(axis=1)

    salary_raw_folded = frame["salary_raw"].map(fold_text)
    raw_has_annual_signal = salary_raw_folded.apply(has_annual_salary_signal)

    frame["salary_period_clean"] = frame["salary_period"].astype("string")
    frame.loc[frame["salary_period_clean"].eq("year") & ~raw_has_annual_signal, "salary_period_clean"] = "month"

    frame["salary_monthly_vnd"] = frame["salary_midpoint"]
    usd_mask = frame["salary_currency"].astype("string").str.upper().eq("USD")
    annual_mask = frame["salary_period_clean"].eq("year") & raw_has_annual_signal
    frame.loc[usd_mask, "salary_monthly_vnd"] = frame.loc[usd_mask, "salary_monthly_vnd"] * USD_TO_VND
    frame.loc[annual_mask, "salary_monthly_vnd"] = frame.loc[annual_mask, "salary_monthly_vnd"] / 12

    frame["salary_monthly_vnd_m"] = frame["salary_monthly_vnd"] / 1_000_000
    frame["log_salary_monthly_vnd"] = np.log(frame["salary_monthly_vnd"].where(frame["salary_monthly_vnd"].gt(0)))

    frame["location_norm"] = frame["location"].map(normalize_location).astype("string")
    frame["skills_list"] = frame["skills"].apply(to_listish)
    frame["skills_norm_list"] = frame["skills_list"].apply(lambda values: sorted({normalize_skill(value) for value in values if normalize_skill(value)}))
    frame["skill_count"] = frame["skills_norm_list"].str.len()

    return frame


def add_top_skill_flags(frame, top_n=20):
    counts = Counter(skill for skills in frame["skills_norm_list"] for skill in skills)
    top_skills = [skill for skill, _ in counts.most_common(top_n)]
    skill_features = []
    for skill in top_skills:
        column = skill_column_name(skill)
        frame[column] = frame["skills_norm_list"].apply(lambda values, selected=skill: int(selected in values))
        skill_features.append(column)
    return top_skills, skill_features

## 2. Load Salary Data

Ta tao dataset cho model luong. Target co 2 phien ban:

- `salary_monthly_vnd_m`: luong thang, don vi trieu VND.
- `log_salary_monthly_vnd`: log cua luong thang VND.

Dung ca 2 de thay vi sao log transform hay duoc dung voi salary data.

In [ ]:
salary = pd.read_csv(SALARY_PATH, encoding="utf-8-sig")
salary_chart = prepare_salary_data(salary)
top_salary_skills, skill_features = add_top_skill_flags(salary_chart, top_n=20)
salary_model_data = salary_chart.dropna(subset=["salary_monthly_vnd_m", "log_salary_monthly_vnd"]).copy()

print(f"Rows: {len(salary_model_data):,}")
print(f"Top skill flags: {len(skill_features)}")
print("Top skills:", top_salary_skills)

summary = salary_model_data["salary_monthly_vnd_m"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(2)
display(summary.to_frame(name="salary_monthly_vnd_m"))

display(
    salary_model_data[["source", "title", "location_norm", "seniority", "work_mode", "experience_min", "skill_count", "salary_monthly_vnd_m"]]
    .head(10)
)

## 3. Linear Regression Pipeline Cho Salary

Cong thuc cua Linear Regression:

```text
y = intercept + coefficient_1 * x_1 + coefficient_2 * x_2 + ...
```

Voi categorical feature, `OneHotEncoder` bien category thanh cot 0/1. Vi du `seniority_senior = 1` neu job la senior, nguoc lai bang 0.

Feature duoc dung:

- Categorical: `source`, `location_norm`, `seniority`, `work_mode`.
- Numeric: `experience_min`, `skill_count`, va top skill flags.

Feature khong dung: cac cot salary goc vi day la leakage.

In [ ]:
categorical_features = ["source", "location_norm", "seniority", "work_mode"]
numeric_features = ["experience_min", "skill_count", *skill_features]
model_features = categorical_features + numeric_features


def build_linear_regression_pipeline():
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_features),
            ("cat", categorical_pipe, categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("linear_regression", LinearRegression()),
        ]
    )


def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def train_salary_model(target_column, random_state=42):
    model_data = salary_model_data.dropna(subset=[target_column]).copy()
    X = model_data[model_features].copy()
    X[categorical_features] = X[categorical_features].astype(object).where(X[categorical_features].notna(), np.nan)
    X[numeric_features] = X[numeric_features].apply(pd.to_numeric, errors="coerce")
    y = model_data[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=random_state,
    )

    model = build_linear_regression_pipeline()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return {
        "model": model,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "y_pred": y_pred,
        "metrics": regression_metrics(y_test, y_pred),
    }


def coefficient_table(model):
    feature_names = model.named_steps["preprocess"].get_feature_names_out()
    coefficients = model.named_steps["linear_regression"].coef_
    return pd.DataFrame({"feature": feature_names, "coefficient": coefficients}).sort_values("coefficient")


def plot_actual_vs_predicted(y_true, y_pred, title, unit_label):
    fig, ax = plt.subplots(figsize=(6, 6))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.75, ax=ax)
    lower = min(float(np.min(y_true)), float(np.min(y_pred)))
    upper = max(float(np.max(y_true)), float(np.max(y_pred)))
    ax.plot([lower, upper], [lower, upper], color="crimson", linestyle="--", label="Perfect prediction")
    ax.set_title(title)
    ax.set_xlabel(f"Actual {unit_label}")
    ax.set_ylabel(f"Predicted {unit_label}")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Model A: Du Bao Salary Goc

Target dau tien la `salary_monthly_vnd_m`, don vi trieu VND/thang.

Model nay de thay Linear Regression bi anh huong nhu the nao khi target co outlier lon.

In [ ]:
raw_salary_result = train_salary_model("salary_monthly_vnd_m")

raw_metrics = pd.Series(raw_salary_result["metrics"]).round(3).to_frame(name="salary_monthly_vnd_m")
display(raw_metrics)

plot_actual_vs_predicted(
    raw_salary_result["y_test"],
    raw_salary_result["y_pred"],
    title="Linear Regression: actual vs predicted salary",
    unit_label="salary, million VND/month",
)

raw_coef = coefficient_table(raw_salary_result["model"])
display(pd.concat([raw_coef.head(10), raw_coef.tail(10)]))

## 5. Model B: Du Bao Log Salary

Target thu hai la `log_salary_monthly_vnd`.

Ly do dung log:

- Salary thuong lech phai.
- Mot vai job luong rat cao co the keo duong hoi quy.
- Log transform lam target gon hon va thuong de Linear Regression hoc hon.

In [ ]:
log_salary_result = train_salary_model("log_salary_monthly_vnd")

log_metrics = pd.Series(log_salary_result["metrics"]).round(3).to_frame(name="log_salary_monthly_vnd")
display(log_metrics)

actual_salary_m = np.exp(log_salary_result["y_test"]) / 1_000_000
predicted_salary_m = np.exp(log_salary_result["y_pred"]) / 1_000_000
back_converted_metrics = pd.Series(regression_metrics(actual_salary_m, predicted_salary_m)).round(3).to_frame(name="back_converted_million_vnd")
display(back_converted_metrics)

plot_actual_vs_predicted(
    actual_salary_m,
    predicted_salary_m,
    title="Linear Regression on log salary: converted back to VND",
    unit_label="salary, million VND/month",
)

log_coef = coefficient_table(log_salary_result["model"])
display(pd.concat([log_coef.head(10), log_coef.tail(10)]))

## 6. Cach Doc Coefficient

Voi target salary goc, coefficient co don vi la trieu VND/thang. Vi du coefficient cua `experience_min` la 5 thi model dang hoc rang them 1 nam kinh nghiem gan voi them khoang 5 trieu VND/thang, khi cac feature khac giu nguyen.

Voi target log salary, coefficient khong doc truc tiep theo trieu VND. Gan dung: coefficient 0.10 co nghia la feature do gan voi salary cao hon khoang 10%.

Day la correlation trong data, khong phai bang chung nhan qua.

## 7. Demand Data: Nhu Cau Tuyen Dung Theo Tuan

Bai toan nhu cau tuyen dung dung target `job_count`. Data duoc lay tu `data/reports/demand_by_week_skill.csv`.

Neu file report chua co, notebook se goi `parsers.trend_reports` de tao tu `data/processed/*_clean.jsonl`.

In [ ]:
def load_or_build_demand_report():
    if not DEMAND_PATH.exists():
        from parsers.trend_reports import default_inputs, write_reports

        print("Demand report not found. Building reports from data/processed/*_clean.jsonl ...")
        write_reports(default_inputs(), REPORTS_DIR)
    return pd.read_csv(DEMAND_PATH, encoding="utf-8-sig")


demand = load_or_build_demand_report()
demand["week_start_dt"] = pd.to_datetime(demand["week_start"], errors="coerce")
demand["skill_norm"] = demand["skill"].map(normalize_skill)
demand["job_count"] = pd.to_numeric(demand["job_count"], errors="coerce")
demand = demand.dropna(subset=["week_start_dt", "job_count"]).copy()

min_week = demand["week_start_dt"].min()
demand["week_index"] = ((demand["week_start_dt"] - min_week).dt.days // 7).astype(int)

skill_totals = demand.loc[demand["skill_norm"].ne("(unknown)")].groupby("skill_norm")["job_count"].sum().sort_values(ascending=False)
top_demand_skills = skill_totals.head(15).index.tolist()
demand_model_data = demand[demand["skill_norm"].isin(top_demand_skills)].copy()

print(f"Demand rows: {len(demand_model_data):,}")
print(f"Weeks: {demand_model_data['week_start_dt'].nunique():,}")
print(f"Date range: {demand_model_data['week_start_dt'].min().date()} -> {demand_model_data['week_start_dt'].max().date()}")
print("Top demand skills:", top_demand_skills)

display(
    demand_model_data.groupby(["week_start", "source"])["job_count"]
    .sum()
    .reset_index()
    .sort_values(["week_start", "source"])
    .head(20)
)

## 8. Linear Regression Cho Demand Trend

Target:

```text
job_count
```

Feature:

- `week_index`: so tuan tinh tu tuan dau tien trong data.
- `source`: nguon job.
- `skill_norm`: skill da chuan hoa.

Neu `week_index` co coefficient duong, model dang hoc xu huong job_count tang theo thoi gian. Neu am, model dang hoc xu huong giam.

Can canh bao: neu data chi co it tuan, day chi la minh hoa Linear Regression, chua phai forecast dang tin.

In [ ]:
demand_numeric_features = ["week_index"]
demand_categorical_features = ["source", "skill_norm"]
demand_features = demand_numeric_features + demand_categorical_features

demand_preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), demand_numeric_features),
        ("cat", Pipeline(steps=[("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), demand_categorical_features),
    ]
)

demand_linear_model = Pipeline(
    steps=[
        ("preprocess", demand_preprocess),
        ("linear_regression", LinearRegression()),
    ]
)

unique_weeks = sorted(demand_model_data["week_start_dt"].dropna().unique())
if len(unique_weeks) < 3:
    print("Canh bao: data co it hon 3 tuan. Split random chi de minh hoa, khong nen xem la forecast that.")
    train_frame, test_frame = train_test_split(demand_model_data, test_size=0.25, random_state=42)
else:
    test_week_count = max(1, round(len(unique_weeks) * 0.25))
    test_weeks = set(unique_weeks[-test_week_count:])
    train_frame = demand_model_data[~demand_model_data["week_start_dt"].isin(test_weeks)].copy()
    test_frame = demand_model_data[demand_model_data["week_start_dt"].isin(test_weeks)].copy()

X_train = train_frame[demand_features]
y_train = train_frame["job_count"]
X_test = test_frame[demand_features]
y_test = test_frame["job_count"]

demand_linear_model.fit(X_train, y_train)
demand_pred = demand_linear_model.predict(X_test)

demand_metrics = pd.Series(regression_metrics(y_test, demand_pred)).round(3).to_frame(name="job_count")
display(demand_metrics)

demand_eval = test_frame[["week_start", "source", "skill_norm", "job_count"]].copy()
demand_eval["predicted_job_count"] = demand_pred
display(demand_eval.sort_values(["week_start", "source", "skill_norm"]).head(20))

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=demand_eval, x="job_count", y="predicted_job_count", hue="source", alpha=0.8, ax=ax)
lower = min(demand_eval["job_count"].min(), demand_eval["predicted_job_count"].min())
upper = max(demand_eval["job_count"].max(), demand_eval["predicted_job_count"].max())
ax.plot([lower, upper], [lower, upper], color="crimson", linestyle="--")
ax.set_title("Demand Linear Regression: actual vs predicted job_count")
ax.set_xlabel("Actual job_count")
ax.set_ylabel("Predicted job_count")
plt.tight_layout()
plt.show()

demand_coef = pd.DataFrame(
    {
        "feature": demand_linear_model.named_steps["preprocess"].get_feature_names_out(),
        "coefficient": demand_linear_model.named_steps["linear_regression"].coef_,
    }
).sort_values("coefficient")
display(pd.concat([demand_coef.head(10), demand_coef.tail(10)]))

## 9. Ket Luan Hoc Duoc Tu Linear Regression

Nhung diem can nho:

- Linear Regression hoc mot duong/thang phang tuyen tinh de giam sai so binh phuong.
- Salary goc co outlier nen Linear Regression de bi keo lech. So sanh model salary goc va log salary se thay dieu nay ro hon.
- Coefficient giup giai thich huong tac dong cua feature, nhung khong chung minh nhan qua.
- Demand `job_count` la count data. Van dung Linear Regression de hoc baseline duoc, nhung khi can forecast that, can nhieu tuan data hon va co the can model rieng cho count/time-series.
- Trong notebook nay, model duy nhat duoc fit la `LinearRegression`.